# 5. Backtest analysis

The long-only EqualWeight-Selected backtest against the 1/N benchmark. Reads the saved predictions and writes the full configuration grid to `Data/Backtest_results/`.

In [ ]:
# Setup - load the two prediction universals
# df_train = the window where hi/K are CHOSEN; df_validation = the held-out window that is CHECKED.
# Default = the 2021-22 (val) / 2023+ (test) split. To run the 2008 crisis experiment, generate the
# crisis parquets with the three Crisis notebooks, then swap the two PATHs + the WIN dict below.
import numpy as np, pandas as pd
DATA_DIR = "Data"

TRAIN_PATH = f"{DATA_DIR}/Predictions_45_val/all_predictions_val.parquet"   # 2021-2022  -> choose hi/K
VALID_PATH = f"{DATA_DIR}/Predictions_45/all_predictions.parquet"           # 2023+      -> held out
# crisis split:
# TRAIN_PATH = f"{DATA_DIR}/Predictions_crisis_val/all_predictions.parquet"   # 2006-2007  (Validation_Crisis)
# VALID_PATH = f"{DATA_DIR}/Predictions_crisis/all_predictions.parquet"       # 2008-2012  (Train_Crisis)

WIN = {"train": ("2021-01-01", "2022-12-31"), "valid": ("2023-01-01", None)}   # default windows
# crisis:  WIN = {"train": ("2006-01-01", "2007-12-31"), "valid": ("2008-01-01", "2012-12-31")}
RESULTS_DIR = f"{DATA_DIR}/Backtest_results"   # the Crisis analysis tab overrides this in its Setup

def read_preds(path):
    d = pd.read_parquet(path)
    d["train_start"] = d["train_start"].astype(str).str[:10]
    d["x_days"] = d["x_days"].astype(int); d["y_up"] = d["y_up"].astype(int)
    return d.reset_index(drop=True)

df_train = read_preds(TRAIN_PATH)
df_validation = read_preds(VALID_PATH)
def win(df): return WIN["train"] if df is df_train else WIN["valid"]

print("df_train", df_train.shape, "| df_validation", df_validation.shape)
print("algos", sorted(df_train.algo.unique()), "| tiers", sorted(df_train.tier.unique()),
      "| starts", sorted(df_train.train_start.unique()))
df_train.head()

In [ ]:
# Features + anchors + a per-slice viewer
# The parquets carry no dates, so predictions map to the first N period anchors rebuilt from the features table:
# daily = every trading day, weekly/monthly = first trading day of each period.
features = pd.concat([pd.read_parquet(f"{DATA_DIR}/features_1.parquet"),
                      pd.read_parquet(f"{DATA_DIR}/features_2.parquet")], ignore_index=True)

def anchors(ticker, scale):
    ft = features[features.ticker == ticker].sort_values("Date")
    if scale == "daily": return ft.reset_index(drop=True)
    key = ft["Date"].dt.to_period("W" if scale == "weekly" else "M")
    return ft.groupby(key, as_index=False).first().sort_values("Date").reset_index(drop=True)

def show(df, ticker, train_start="2000-01-01", x_days=30, algo="lstm", scale="weekly", fusion=None):
    if fusion is None: fusion = "single" if scale == "daily" else "together"
    m = ((df.ticker==ticker)&(df.algo==algo)&(df.tier=="gmm")&(df.scale==scale)&(df.fusion==fusion)
         &(df.train_start==train_start)&(df.x_days==x_days))
    sel = df[m].reset_index(drop=True)
    start, end = win(df)
    a = anchors(ticker, scale); a = a[a.Date >= pd.Timestamp(start)]
    if end: a = a[a.Date <= pd.Timestamp(end)]
    a = a.reset_index(drop=True); n = min(len(sel), len(a)); out = sel.iloc[:n].copy()
    for c in ["Date","Open","High","Low","Close"]: out[c] = a[c].to_numpy()[:n]
    keep = ["Date","ticker","prob_up","y_up"] + (["regime"] if "regime" in out.columns else [])
    return out[keep]

show(df_train, "AAPL", scale="weekly", fusion="split", train_start="2015-01-01", x_days=100)

In [ ]:
# Capital / fee, the 1/N benchmark, and a max-drawdown helper
# 1/N = buy an equal dollar of all 40 on the first trading day and hold to the last (scale-independent,
# pays the fee once). Both the strategy and 1/N start at $100k, so end value / return% / MDD compare directly.
CAPITAL, FEE = 100000.0, 0.0005     # 5 bps per trade (IBKR-realistic); FEE is a global read by every function

def portfolio_1overN(df):
    start, end = win(df); op, cl = {}, {}
    for tk in sorted(df.ticker.unique()):
        d = features[features.ticker==tk].sort_values("Date"); d = d[d.Date >= pd.Timestamp(start)]
        if end: d = d[d.Date <= pd.Timestamp(end)]
        op[tk] = pd.Series(d["Open"].to_numpy(),  index=d["Date"].to_numpy())
        cl[tk] = pd.Series(d["Close"].to_numpy(), index=d["Date"].to_numpy())
    op = pd.DataFrame(op).sort_index(); cl = pd.DataFrame(cl).sort_index()
    shares = (CAPITAL/len(op.columns)) / op.iloc[0]
    return cl.mul(shares, axis=1).sum(axis=1) - FEE*CAPITAL

def mdd_of(v):
    v = np.asarray(v, float); return round(float((1 - v/np.maximum.accumulate(v)).max()*100), 2)

print(f"1/N: train {portfolio_1overN(df_train).iloc[-1]:,.0f} | validation {portfolio_1overN(df_validation).iloc[-1]:,.0f}")

In [ ]:
# The gate fit on TRAIN - per-ticker buy threshold hi, and the tradable regimes
# fit_hi sweeps a buy/sell band per ticker and keeps the hi of the best acted-accuracy band (coverage >= min_cov).
# build_trad_cfg marks a regime tradable if its train directional accuracy >= min_acc with >= min_obs obs.
# Both are always fit on df_train and then applied to whichever window is backtested (kept out-of-sample).
def fit_hi(df, cfg, by="acc_acted", min_cov=0.3,
           his=np.round(np.arange(0.50,0.86,0.02),3), los=np.round(np.arange(0.15,0.51,0.02),3)):
    m = np.logical_and.reduce([df[c]==v for c,v in cfg.items() if c in df.columns]); sub=df[m]; hi_map={}
    for tk in sorted(sub.ticker.unique()):
        s=sub[sub.ticker==tk]; y=s["y_up"].to_numpy(); p=s["prob_up"].to_numpy(); best=None
        for hi in his:
            for lo in los:
                if lo>=hi: continue
                buy=p>=hi; sell=p<=lo; acted=buy|sell; cov=acted.mean()
                if cov<min_cov or not acted.any(): continue
                acc=((buy&(y==1))|(sell&(y==0)))[acted].mean(); edge=cov*(acc-0.5)
                score=acc if by=="acc_acted" else edge
                if best is None or (score,edge)>best[:2]: best=(score,edge,hi)
        if best is not None: hi_map[tk]=best[2]
    return hi_map

def build_trad_cfg(df, cfg, min_acc=0.50, min_obs=20):
    m = np.logical_and.reduce([df[c]==v for c,v in cfg.items() if c in df.columns]); sub=df[m]; trad={}
    for tk in sorted(sub.ticker.unique()):
        s=sub[sub.ticker==tk]
        if "regime" not in s.columns or s["regime"].isna().all(): trad[tk]=set(); continue
        d=s.dropna(subset=["regime"]).copy(); d["hit"]=((d.prob_up>0.5)==(d.y_up==1))
        g=d.groupby(d.regime).agg(n=("hit","size"), acc=("hit","mean"))
        trad[tk]=set(g.index[(g.acc>=min_acc)&(g.n>=min_obs)])
    return trad

In [ ]:
# Per-ticker signals + daily bars for one config on one window
# elig = prob >= its own hi (and, with GMM on, regime is tradable). Signals sit on the anchor dates;
# daily OHLC bars run over the window for the intraday TP/SL check.
def tpsl_data(df, cfg, hi_map, trad=None, use_regime=False):
    start, end = win(df)
    m = np.logical_and.reduce([df[c]==v for c,v in cfg.items() if c in df.columns]); sub=df[m]
    tickers=sorted(sub.ticker.unique()); sig, daily = {}, {}
    for tk in tickers:
        s=sub[sub.ticker==tk].reset_index(drop=True)
        aw=anchors(tk,cfg["scale"]); aw=aw[aw.Date>=pd.Timestamp(start)]
        if end: aw=aw[aw.Date<=pd.Timestamp(end)]
        aw=aw.reset_index(drop=True); n=min(len(s),len(aw))
        prob=s["prob_up"].to_numpy()[:n]; reg=s["regime"].to_numpy()[:n] if "regime" in s.columns else np.full(n,np.nan)
        elig=prob>=hi_map.get(tk,np.inf)
        if use_regime and trad is not None:
            tr=trad.get(tk)
            if tr is not None: elig=elig & np.isin(reg,list(tr))
        sig[tk]=pd.DataFrame({"prob":prob,"regime":reg,"elig":elig}, index=pd.to_datetime(aw["Date"].to_numpy()[:n]))
        d=features[features.ticker==tk].sort_values("Date"); d=d[d.Date>=pd.Timestamp(start)]
        if end: d=d[d.Date<=pd.Timestamp(end)]
        daily[tk]=d.set_index("Date")[["Open","High","Low","Close"]]
    return tickers, sig, daily

In [ ]:
# Pack one config into numpy arrays for the fast engine (K-independent, so prep once and sweep K)
# NEXT-BAR EXECUTION: each anchor's signal is placed on the FOLLOWING trading day (di[d]+1), so the engine
# trades at the open of the day AFTER the signal is known - the prediction uses the anchor day's close, so
# entering at that day's open would be a look-ahead. The last anchor is dropped (no next bar to trade it).
def tpsl_prep(df, cfg, hi_map, trad=None, use_regime=False):
    tickers, sig, daily = tpsl_data(df, cfg, hi_map, trad, use_regime)
    days=pd.DatetimeIndex(sorted(set().union(*[set(daily[tk].index) for tk in tickers]))); di={d:i for i,d in enumerate(days)}; N=len(days)
    O={};H={};L={};C={}
    for tk in tickers:
        d=daily[tk].reindex(days); O[tk]=d.Open.to_numpy();H[tk]=d.High.to_numpy();L[tk]=d.Low.to_numpy();C[tk]=d.Close.to_numpy()
    rebal=sorted({di[d]+1 for tk in tickers for d in sig[tk].index if d in di and di[d]+1 < N})
    prob_at={i:{} for i in rebal}; elig_at={i:{} for i in rebal}
    for tk in tickers:
        p=sig[tk]["prob"].to_numpy(); e=sig[tk]["elig"].to_numpy()
        for j,d in enumerate(sig[tk].index):
            if d in di and di[d]+1 < N: i=di[d]+1; prob_at[i][tk]=p[j]; elig_at[i][tk]=bool(e[j])
    return dict(days=days,O=O,H=H,L=L,C=C,rebal=set(rebal),prob_at=prob_at,elig_at=elig_at)

In [ ]:
# The fast backtest (grid engine) + a one-line helper
# EW-Selected + TP/SL: at each rebalance buy top-K eligible names by prob (free cash / new names, at the open);
# daily TP/SL on High/Low with SL winning ties; hold otherwise, sell at a rebalance when the signal dies.
# Equity is marked DAILY, so the max drawdown is comparable to the 1/N benchmark. Returns (end_capital, mdd%).
def tpsl_run(P, K, tp=0.25, sl=0.15):
    O,H,L,C,rebal,prob_at,elig_at = P["O"],P["H"],P["L"],P["C"],P["rebal"],P["prob_at"],P["elig_at"]
    cash=CAPITAL; pos={}; V=[CAPITAL]
    for i in range(len(P["days"])):
        for tk in list(pos):
            lo=L[tk][i]
            if lo!=lo: continue
            sh,e=pos[tk]
            if   lo<=e*(1-sl): fill=e*(1-sl)
            elif H[tk][i]>=e*(1+tp): fill=e*(1+tp)
            else: continue
            cash+=sh*fill*(1-FEE); del pos[tk]
        if i in rebal:
            ea,pa=elig_at[i],prob_at[i]
            for tk in list(pos):
                if tk in ea and not ea[tk]: cash+=pos[tk][0]*O[tk][i]*(1-FEE); del pos[tk]
            slots=K-len(pos)
            if slots>0:
                cand=[tk for tk in ea if ea[tk] and tk not in pos]; cand.sort(key=lambda tk:-pa[tk])
                buy=cand[:slots]
                if buy:
                    each=cash/len(buy)
                    for tk in buy:
                        px=O[tk][i]; pos[tk]=((each/(1+FEE))/px, px); cash-=each
        V.append(cash + sum(sh*C[tk][i] for tk,(sh,e) in pos.items() if C[tk][i]==C[tk][i]))
    end=V[-1]; V=np.array(V); mdd=float((1 - V/np.maximum.accumulate(V)).max()*100)
    return round(end), round(mdd,2)

def ew_tpsl_fast(df, cfg, use_regime=False):
    hi_map=fit_hi(df_train,cfg); trad=build_trad_cfg(df_train,cfg)      # thresholds always from TRAIN
    return tpsl_run(tpsl_prep(df,cfg,hi_map,trad,use_regime), cfg["k"])

In [ ]:
# The logging engine for ONE config - full trade log (tx) + daily equity (eq)
# Same rules as tpsl_run (next-bar execution, daily equity so drawdown matches 1/N). Pass tp=0.25, sl=0.15
# to reconcile with the grid to the dollar. Prints end / PnL / realized / fees and the strategy-vs-1/N drawdown.
def ew_tpsl(df, cfg, use_regime=False, tp=0.25, sl=0.15):
    K = cfg["k"]
    hi_map = fit_hi(df_train, cfg); trad = build_trad_cfg(df_train, cfg)
    tickers, sig, daily = tpsl_data(df, cfg, hi_map, trad, use_regime)
    all_days = sorted(set().union(*[set(daily[tk].index) for tk in tickers]))
    pos_of = {d:i for i,d in enumerate(all_days)}
    exec_sig = {}
    for tk in tickers:
        for a in sig[tk].index:
            if a in pos_of and pos_of[a]+1 < len(all_days):
                ed = all_days[pos_of[a]+1]
                exec_sig.setdefault(ed, {})[tk] = (float(sig[tk].loc[a,"prob"]), bool(sig[tk].loc[a,"elig"]))
    rebal = set(exec_sig)
    cash = CAPITAL; pos = {}; tx = []; eq = []
    for t in all_days:
        for tk in list(pos):
            if t not in daily[tk].index: continue
            bar = daily[tk].loc[t]; e = pos[tk]["entry"]; sh = pos[tk]["shares"]
            hit = fill = None
            if bar["Low"]  <= e*(1-sl): hit, fill = "SL", e*(1-sl)
            elif bar["High"] >= e*(1+tp): hit, fill = "TP", e*(1+tp)
            if hit:
                gross = sh*fill; fee = FEE*gross; cash += gross-fee
                tx.append({"Date":t.date(),"ticker":tk,"side":"SELL","reason":hit,"shares":round(sh,3),
                           "price":round(fill,2),"fee":round(fee,2),"pnl":round(sh*(fill-e)-fee,2)}); del pos[tk]
        if t in rebal:
            sigt = exec_sig[t]
            for tk in list(pos):
                if tk in sigt and not sigt[tk][1] and t in daily[tk].index:
                    px = float(daily[tk].loc[t,"Open"]); sh = pos[tk]["shares"]; e = pos[tk]["entry"]
                    gross = sh*px; fee = FEE*gross; cash += gross-fee
                    tx.append({"Date":t.date(),"ticker":tk,"side":"SELL","reason":"signal","shares":round(sh,3),
                               "price":round(px,2),"fee":round(fee,2),"pnl":round(sh*(px-e)-fee,2)}); del pos[tk]
            slots = K - len(pos)
            if slots > 0:
                cand = [tk for tk in sigt if sigt[tk][1] and tk not in pos and t in daily[tk].index]
                cand.sort(key=lambda tk: -sigt[tk][0]); buy = cand[:slots]
                if buy:
                    each = cash/len(buy)
                    for tk in buy:
                        px = float(daily[tk].loc[t,"Open"]); sh = (each/(1+FEE))/px
                        gross = sh*px; fee = FEE*gross; cash -= gross+fee; pos[tk] = {"shares":sh,"entry":px}
                        tx.append({"Date":t.date(),"ticker":tk,"side":"BUY","reason":"entry","shares":round(sh,3),
                                   "price":round(px,2),"fee":round(fee,2),"pnl":np.nan})
        val = cash + sum(pos[tk]["shares"]*float(daily[tk].loc[t,"Close"]) for tk in pos if t in daily[tk].index)
        eq.append({"Date":t.date(),"value":round(val,0),"cash":round(cash,0),"n_held":len(pos),"held":", ".join(pos)})
    end_val = cash + sum(pos[tk]["shares"]*float(daily[tk]["Close"].iloc[-1]) for tk in pos)
    unreal  = sum(pos[tk]["shares"]*(float(daily[tk]["Close"].iloc[-1])-pos[tk]["entry"]) for tk in pos)
    tx = pd.DataFrame(tx); eq = pd.DataFrame(eq)
    eq["pnl_$"]      = eq["value"].diff().round(0)
    eq["cum_ret_%"]  = ((eq["value"]/CAPITAL-1)*100).round(2)
    eq["drawdown_%"] = ((eq["value"]/eq["value"].cummax()-1)*100).round(2)
    realized = tx[tx.side=="SELL"].pnl.sum() if len(tx) else 0.0
    strat_mdd = (1 - eq["value"]/eq["value"].cummax()).max()*100 if len(eq) else 0.0
    onen_s = portfolio_1overN(df); onen = onen_s.iloc[-1]; onen_mdd = mdd_of(onen_s)
    scen = "val" if df is df_validation else "train"
    print(f"EW+TPSL K={K} | {scen} | GMM={use_regime}: end {end_val:,.0f} | PnL {end_val-CAPITAL:,.0f} | "
          f"realized {realized:,.0f} | unrealized {unreal:,.0f} | fees {tx.fee.sum() if len(tx) else 0:,.0f} | trades {len(tx)}")
    print(f"1/N   {scen}: end {onen:,.0f} | PnL {onen-CAPITAL:,.0f}   ->   EW+TPSL "
          f"{'beats' if end_val>onen else 'loses to'} 1/N by {end_val-onen:,.0f}")
    print(f"max drawdown: EW+TPSL {strat_mdd:.1f}%  |  1/N {onen_mdd:.1f}%")
    return tx, eq

In [ ]:
# The grid - sweep every config x K on TRAIN and VALIDATION, save to parquet with 1% checkpoint + resume
# Count = 3 algo x 5 mode x 3 start x 4 x_days x 2 tier x 2 gmm = 720 configs x 40 K = 28,800 rows.
# STARTS auto-derives from the data; RESULTS_DIR comes from the Setup cell (Backtest_results, or Backtest_results_crisis in the Crisis tab).
import os
ALGOS=["lstm","xgb","knn"]; SCALES=["daily","weekly","monthly"]
FUSIONS=["together","split"]; STARTS=sorted(df_train["train_start"].unique()); XDAYS=[10,20,30,100]
TIERS=["gmm","pooled"]; GMMS=[False,True]; KS=list(range(1,41))
def fusions_for(sc): return ["single"] if sc=="daily" else FUSIONS
total_cfg=len(ALGOS)*sum(len(fusions_for(sc)) for sc in SCALES)*len(STARTS)*len(XDAYS)*len(TIERS)*len(GMMS)

os.makedirs(RESULTS_DIR, exist_ok=True)
path=f"{RESULTS_DIR}/results_all.parquet"
cfg_cols=["algo","tier","scale","fusion","train_start","x_days","gmm","k"]
key_cols=["algo","tier","scale","fusion","train_start","x_days","gmm"]
COLS=cfg_cols+["end_capital_train","1/N_train","end_capital_validation","1/N_validation",
               "mdd_train","1/N_mdd_train","mdd_validation","1/N_mdd_validation"]

v1_tr=portfolio_1overN(df_train); v1_va=portfolio_1overN(df_validation)
onen_tr, onen_va = round(v1_tr.iloc[-1]), round(v1_va.iloc[-1])
onen_mdd_tr, onen_mdd_va = mdd_of(v1_tr), mdd_of(v1_va)

if os.path.exists(path):
    prev=pd.read_parquet(path); rows=prev.to_dict("records")
    done_set=set(map(tuple, prev[key_cols].values.tolist()))
    print(f"resuming: {len(done_set)}/{total_cfg} configs already done ({len(rows):,} rows)")
else:
    rows=[]; done_set=set()

def save():
    tmp=path+".tmp"; pd.DataFrame(rows)[COLS].to_parquet(tmp); os.replace(tmp, path)

print(f"1/N: train {onen_tr:,} (MDD {onen_mdd_tr}%) | val {onen_va:,} (MDD {onen_mdd_va}%) | {total_cfg} configs")
done=len(done_set); mark=int(done/total_cfg*100)+1
for ALGO in ALGOS:
  for SCALE in SCALES:
    for fusion in fusions_for(SCALE):
      for ts in STARTS:
        for xd in XDAYS:
          for tier in TIERS:
            if all((ALGO,tier,SCALE,fusion,ts,xd,g) in done_set for g in GMMS): continue
            cfg=dict(algo=ALGO,tier=tier,scale=SCALE,fusion=fusion,train_start=ts,x_days=xd)
            hi_map=fit_hi(df_train,cfg); trad=build_trad_cfg(df_train,cfg)
            for gmm in GMMS:
                key=(ALGO,tier,SCALE,fusion,ts,xd,gmm)
                if key in done_set: continue
                base={**cfg,"gmm":gmm}
                Ptr=tpsl_prep(df_train,cfg,hi_map,trad,use_regime=gmm)
                Pva=tpsl_prep(df_validation,cfg,hi_map,trad,use_regime=gmm)
                for k in KS:
                    et,dt=tpsl_run(Ptr,k); ev,dv=tpsl_run(Pva,k)
                    rows.append({**base,"k":k,
                                 "end_capital_train":et,"1/N_train":onen_tr,
                                 "end_capital_validation":ev,"1/N_validation":onen_va,
                                 "mdd_train":dt,"1/N_mdd_train":onen_mdd_tr,
                                 "mdd_validation":dv,"1/N_mdd_validation":onen_mdd_va})
                done_set.add(key); done+=1
                while done/total_cfg*100>=mark and mark<=100:
                    save(); print(f"{done}/{total_cfg} done, {mark}% (saved)"); mark+=1

save()
M_all=pd.read_parquet(path)
print("done. saved", M_all.shape, "->", path)
M_all.sort_values("end_capital_validation", ascending=False).head(20)

In [ ]:
# Read the saved grid, pick the best config on VALIDATION, inspect it with the logging engine
def row_to_cfg(row):
    cfg = {c: row[c] for c in ["algo","tier","scale","fusion","train_start","x_days"]}
    cfg["x_days"] = int(cfg["x_days"]); cfg["k"] = int(row["k"])
    return cfg, bool(row["gmm"])

M_all = pd.read_parquet(f"{RESULTS_DIR}/results_all.parquet")
best = M_all.sort_values("end_capital_validation", ascending=False).iloc[0]
CFG_KW, use_regime = row_to_cfg(best)
print(CFG_KW, "| use_regime =", use_regime)
tx, eq = ew_tpsl(df_validation, CFG_KW, use_regime=use_regime, tp=0.25, sl=0.15)
eq.head(20)

In [ ]:
# Mega portfolio - combine N configs into one $100k book (split equally), vs 1/N
# Choose the best N from the saved grid (pick_best_n) or hand-write a CFGS list; run on train or validation,
# fast (summary + eq) or detail=True (also the full trade log tx). Thresholds are still fit on train.
def pick_best_n(n, distinct=True, by="end_capital_train"):
    M = pd.read_parquet(f"{RESULTS_DIR}/results_all.parquet").sort_values(by, ascending=False)
    if distinct: M = M.drop_duplicates(["algo","tier","scale","fusion","train_start","x_days","gmm"])
    return [dict(algo=r.algo,tier=r.tier,scale=r.scale,fusion=r.fusion,train_start=r.train_start,
                 x_days=int(r.x_days), k=int(r.k), use_regime=bool(r.gmm)) for _,r in M.head(n).iterrows()]

def sub_run(df, cfg, capital, log=False, tag="", tp=0.25, sl=0.15):
    K=cfg["k"]; ur=cfg.get("use_regime",False)
    mcfg={c:cfg[c] for c in ["algo","tier","scale","fusion","train_start","x_days"]}
    hi_map=fit_hi(df_train,mcfg); trad=build_trad_cfg(df_train,mcfg)
    P=tpsl_prep(df,mcfg,hi_map,trad,use_regime=ur)
    O,H,L,C,rebal,prob_at,elig_at,days=P["O"],P["H"],P["L"],P["C"],P["rebal"],P["prob_at"],P["elig_at"],P["days"]
    cash=capital; pos={}; V=[]; tx=[] if log else None
    for i in range(len(days)):
        t=pd.Timestamp(days[i])
        for tk in list(pos):
            lo=L[tk][i]
            if lo!=lo: continue
            sh,e=pos[tk]
            if   lo<=e*(1-sl): reason,fill="SL",e*(1-sl)
            elif H[tk][i]>=e*(1+tp): reason,fill="TP",e*(1+tp)
            else: continue
            fee=FEE*sh*fill; cash+=sh*fill-fee
            if log: tx.append({"strategy":tag,"Date":t.date(),"ticker":tk,"side":"SELL","reason":reason,"shares":round(sh,3),"price":round(fill,2),"fee":round(fee,2),"pnl":round(sh*(fill-e)-fee,2)})
            del pos[tk]
        if i in rebal:
            ea,pa=elig_at[i],prob_at[i]
            for tk in list(pos):
                if tk in ea and not ea[tk]:
                    sh,e=pos[tk]; px=O[tk][i]; fee=FEE*sh*px; cash+=sh*px-fee
                    if log: tx.append({"strategy":tag,"Date":t.date(),"ticker":tk,"side":"SELL","reason":"signal","shares":round(sh,3),"price":round(px,2),"fee":round(fee,2),"pnl":round(sh*(px-e)-fee,2)})
                    del pos[tk]
            slots=K-len(pos)
            if slots>0:
                cand=[tk for tk in ea if ea[tk] and tk not in pos]; cand.sort(key=lambda tk:-pa[tk]); buy=cand[:slots]
                if buy:
                    each=cash/len(buy)
                    for tk in buy:
                        px=O[tk][i]; sh=(each/(1+FEE))/px; fee=FEE*sh*px; cash-=sh*px+fee; pos[tk]=(sh,px)
                        if log: tx.append({"strategy":tag,"Date":t.date(),"ticker":tk,"side":"BUY","reason":"entry","shares":round(sh,3),"price":round(px,2),"fee":round(fee,2),"pnl":np.nan})
        V.append(cash + sum(sh*C[tk][i] for tk,(sh,e) in pos.items() if C[tk][i]==C[tk][i]))
    return pd.Series(V, index=pd.DatetimeIndex(days)), (pd.DataFrame(tx) if log else None)

def mega(cfgs, df, detail=False, total=100000):
    cap=total/len(cfgs); curves=[]; txs=[]
    for j,c in enumerate(cfgs):
        tag=f"#{j+1} {c['algo']}/{c['tier']}/{c['scale']}/{c['fusion']}/{c['train_start'][:4]}/x{c['x_days']}/gmm={c.get('use_regime',False)}/K={c['k']}"
        v,tx=sub_run(df,c,cap,log=detail,tag=tag); curves.append(v)
        if detail: txs.append(tx)
    days=pd.DatetimeIndex(sorted(set().union(*[set(v.index) for v in curves])))
    eq=pd.DataFrame({f"strat{j+1}":curves[j].reindex(days).ffill().fillna(cap) for j in range(len(curves))}, index=days)
    eq["mega"]=eq[[f"strat{j+1}" for j in range(len(curves))]].sum(axis=1)
    eq["cum_ret_%"]=((eq["mega"]/total-1)*100).round(2)
    eq["drawdown_%"]=((eq["mega"]/eq["mega"].cummax()-1)*100).round(2)
    v1=portfolio_1overN(df); scen="val" if df is df_validation else "train"
    print(f"[{scen}] MEGA {len(cfgs)}x{cap:,.0f}: end {eq['mega'].iloc[-1]:,.0f} | ret {eq['cum_ret_%'].iloc[-1]:+.1f}% | maxDD {-eq['drawdown_%'].min():.1f}%"
          f"  ||  1/N: end {v1.iloc[-1]:,.0f} | ret {(v1.iloc[-1]/100000-1)*100:+.1f}% | maxDD {(1-v1/v1.cummax()).max()*100:.1f}%")
    return (pd.concat(txs,ignore_index=True).sort_values("Date").reset_index(drop=True), eq) if detail else eq

CFGS = pick_best_n(4)                                # or hand-write a list of CFG dicts
eq = mega(CFGS, df_validation)                       # fast; mega(CFGS, df_validation, detail=True) -> (tx, eq)

In [ ]:
# Train -> validation transfer (the honest check) + the risk-return picture
# Reads results_all.parquet: how many configs beat 1/N on each window, the rank-correlation of train vs
# validation end-capital (all rows + within-K), whether the top-20 train picks survive on validation, and
# the risk-return scatter with 1/N marked. Near-zero within-K correlation + top-20 at the base rate = no transfer.
import matplotlib.pyplot as plt
M = pd.read_parquet(f"{RESULTS_DIR}/results_all.parquet")
M["beat_tr"]=M.end_capital_train>M["1/N_train"]; M["beat_va"]=M.end_capital_validation>M["1/N_validation"]
print(f"{len(M):,} rows | beat 1/N: train {M.beat_tr.mean()*100:.0f}% | validation {M.beat_va.mean()*100:.0f}%")
sp=M.end_capital_train.corr(M.end_capital_validation, method="spearman")
wk=[g.end_capital_train.corr(g.end_capital_validation,method='spearman') for _,g in M.groupby('k')]
print(f"train vs validation Spearman {sp:.3f} (all rows) | within-K mean {np.nanmean(wk):.3f}")
top=M.sort_values("end_capital_train",ascending=False).head(20)
print(f"top-20 on train -> {top.beat_va.mean()*100:.0f}% beat 1/N on validation (base rate {M.beat_va.mean()*100:.0f}%)")

onen_tr,onen_va=M["1/N_train"].iloc[0],M["1/N_validation"].iloc[0]
fig,ax=plt.subplots(1,2,figsize=(13,5.5))
ax[0].scatter(M.end_capital_train,M.end_capital_validation,s=7,alpha=.25)
ax[0].axvline(onen_tr,ls="--",c="g",label="1/N train"); ax[0].axhline(onen_va,ls="--",c="r",label="1/N validation")
ax[0].set_xlabel("train end capital"); ax[0].set_ylabel("validation end capital")
ax[0].set_title(f"train vs validation (Spearman {sp:.2f})"); ax[0].legend()
M["ret_va"]=M.end_capital_validation/1e5-1
for sc,c in [("daily","#888"),("weekly","#1f77b4"),("monthly","#d62728")]:
    s=M[M.scale==sc]; ax[1].scatter(s.mdd_validation,s.ret_va*100,s=7,alpha=.25,c=c,label=sc)
ax[1].scatter([M["1/N_mdd_validation"].iloc[0]],[(onen_va/1e5-1)*100],marker="*",s=350,c="gold",edgecolor="k",zorder=5,label="1/N")
ax[1].set_xlabel("max drawdown % (validation)"); ax[1].set_ylabel("return % (validation)")
ax[1].set_title("risk vs return"); ax[1].legend()
plt.tight_layout(); plt.show()